# 🧩 Extensibility & Custom Module Plugins in NeuralOps

In this tutorial, we register custom attention mechanisms and custom composable stages into the NeuralOps runtime, run the automated 6-point contract diagnostic suite, and scaffold new operators.

In [ ]:
!pip install neuralops torch
import torch
import torch.nn as nn
import torch.nn.functional as F
import neuralops

print(f"NeuralOps Version: {neuralops.__version__}")

## 1. Register Custom Attention Mechanism
Decorate any standard PyTorch `nn.Module` with `@neuralops.register_custom_attention`.

In [ ]:
@neuralops.register_custom_attention(
    "my_research_attn",
    family="linear-scan",
    time_complexity="O(T·d²)",
    memory_complexity="O(d²) state",
    math_equation="S_t = S_{t-1} + k_t^T @ v_t | o_t = q_t @ S_t",
)
class MyResearchAttention(nn.Module):
    def __init__(self, embed_dim: int = 64, num_heads: int = 4, **kwargs):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.v = nn.Linear(embed_dim, embed_dim, bias=False)
        self.out = nn.Linear(embed_dim, embed_dim, bias=False)
    def forward(self, x, *args, **kwargs):
        B, T, D = x.shape
        H, d = self.num_heads, self.head_dim
        q = self.q(x).view(B, T, H, d).transpose(1, 2)
        k = self.k(x).view(B, T, H, d).transpose(1, 2)
        v = self.v(x).view(B, T, H, d).transpose(1, 2)
        scores = torch.matmul(F.elu(q) + 1.0, (F.elu(k) + 1.0).transpose(-1, -2))
        ctx = torch.matmul(scores, v)
        return self.out(ctx.transpose(1, 2).contiguous().view(B, T, D))

print("Registered 'my_research_attn' successfully!")

## 2. Automated 6-Point Contract Diagnostic Validation

In [ ]:
val_results = neuralops.validate_module_contract("my_research_attn", embed_dim=64, num_heads=4)
print(f"All Passed: {val_results['all_passed']} ({val_results['passed_count']}/{val_results['total_checks']})")
for check, res in val_results["checks"].items():
    print(f"  [✓] {check}: {res['status']}")

## 3. Register Custom Composable Algebraic Kernel

In [ ]:
@neuralops.register_composable_kernel("custom_gelu_map")
def custom_gelu_map(x: torch.Tensor) -> torch.Tensor:
    return F.gelu(x) + 1.0

composed = neuralops.compose_attention(
    projection="dense",
    kernel_map="custom_gelu_map",
    mask="causal",
    engine="sdpa",
    embed_dim=64,
    num_heads=4,
)

x = torch.randn(2, 16, 64)
out = composed(x)
print(f"Composed Attention output shape: {out.shape}")